# Approximate Earth with Activated SH

Code from Tess Smidt, Ameya Daigavane, and YuQing Xie

In [ ]:
!pip install e3nn

In [ ]:
import torch
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from e3nn import io, o3

import torch.nn.functional as F

import numpy as np

import matplotlib.pyplot as plt

## Import and plot target signal

In [ ]:
from PIL import Image
import numpy as np

res = 100

# Open the image file
with Image.open('earth_image.png') as img:
    # Resize the image
    img_resized = img.resize((res-1, res))  # Width, Height

    # Convert the image to a NumPy array
    img_gray = img_resized.convert('L')

    # Convert the grayscale image to a NumPy array
    img_array = np.array(img_gray)

    # Optionally, display the image array dimensions and data type
    print("Array Shape:", img_array.shape)
    print("Data Type:", img_array.dtype)

    # Now img_array can be used for further image processing


In [ ]:
img_array_inv = (255 - img_array)/255

In [ ]:
plt.imshow(img_array_inv, cmap="Greys")

## Plot image on sphere

In [ ]:
import plotly.graph_objects as go
import numpy as np
from PIL import Image

# Load and preprocess image
# img = Image.open("earth_image.png").convert("L")
# data = np.array(img)
# data = data / 255.0  # normalize to [0, 1]

data = 1 - img_array / 255

data = data[::-1]

# Get lat/lon dimensions
lat_count, lon_count = data.shape
lats = np.linspace(-np.pi / 2, np.pi / 2, lat_count)
lons = np.linspace(0, 2 * np.pi, lon_count)

# Create spherical coordinates (radius = 1)
lat_grid, lon_grid = np.meshgrid(lats, lons, indexing='ij')
x = np.cos(lat_grid) * np.cos(lon_grid)
y = np.cos(lat_grid) * np.sin(lon_grid)
z = np.sin(lat_grid)

# Plot sphere with image as surface color
fig = go.Figure(data=[go.Surface(
    x=x, y=y, z=z,
    surfacecolor=data,
    colorscale='Greys',
    cmin=0, cmax=1,
    showscale=False
)])

fig.show()



## Plot bandwidth limited signal on sphere up to $L_{max}=6$

In [ ]:
lmax = 10
res = 100

ir = o3.Irreps.spherical_harmonics(lmax=lmax)

fs2 = o3.FromS2Grid(res=res, lmax=lmax)

# Project image array onto Spherical harmonics up to lmax
target = torch.Tensor(img_array_inv).to(torch.float32)
sig = fs2(target)

# If we want to rotate so north pole is along z axis (instead of along y)
# This is just to match above
rot_y_to_z = ir.D_from_axis_angle(torch.tensor([1., 0, 0]), torch.tensor(np.pi/2))
rot_x_to_y = ir.D_from_axis_angle(torch.tensor([0., 0, 1.]), torch.tensor(np.pi/2))
plot_sig = rot_x_to_y @ rot_y_to_z @ sig

ts2 = o3.ToS2Grid(res=res, lmax=lmax)

gridded_sig = ts2(sig)

sphten = io.SphericalTensor(lmax=lmax, p_arg=-1, p_val=1)

surf = go.Surface(**sphten.plotly_surface(plot_sig, radius=False, res=res)[0])
surf.colorscale = 'Greys'

fig = go.Figure([
    surf
])

# fig.write_html("earth_{}.html".format(lmax), auto_open=False)
fig.show()

## Plot activation function (Scaled Exponential Linear Unit) SELU

In [ ]:
x = torch.linspace(-5, 5, 100)
y = F.selu(x)

fig, ax = plt.subplots(1, 1, figsize=(3, 2))

ax.plot(x, y)

## N copies of Spherical Harmonics $\rightarrow$ SELU activation $\rightarrow$ Scalar Linear layer on sphere
1. Start with $N$ random spherical harmonic (SH) coefficients up to $L_{max} = 6$, a total of $N \times (L_{max} + 1)^2$ coefficients.
2. Evaluate coefficients on sphere to build $N$ scalar signals on sphere.
3. Apply $N \times 1$ linear layer to mix $N \rightarrow 1$ scalar fields
4. Compute loss between output and target signal
5. Backpropogate to update input SH coefficients

### Set up parameters, grid, linear layer, and loss

In [ ]:
N = 20
res = (100, 99)

target = torch.Tensor(img_array_inv).to(torch.float32)

torch.manual_seed(42)
x = torch.randn(N, (lmax + 1)**2)
x.requires_grad = True

t_s2 = o3.ToS2Grid(lmax=lmax, res=res)
mse = torch.nn.MSELoss()

lin = torch.nn.Linear(N, 1, bias=True)

### Train! (Evaluate loss and backpropogate)

In [ ]:
max_iter = 5001
opt = torch.optim.Adam([x]+ list(lin.parameters()), lr=3e-2)
for i in range(max_iter):
    on_sphere = t_s2(x)
    on_sphere = on_sphere.reshape(N, -1)
    s2_out = F.selu(on_sphere)
    lin_out = lin(s2_out.transpose(1, 0))
    out = lin_out.transpose(1, 0)
    out = out.reshape(*res)
    loss = mse(out, target)
    if i % 500 == 0:
        print(i, loss.detach().numpy())
    opt.zero_grad()
    loss.backward()
    opt.step()

### Visualize

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3)
vmax = 1
vmin = -vmax
ax[0].imshow(target, cmap="RdBu", vmin=vmin, vmax=vmax)
# ax[1].imshow(np.log(out.detach()), cmap="Greys")
ax[1].imshow(gridded_sig, cmap="RdBu", vmin=vmin, vmax=vmax)
ax[2].imshow(out.detach(), cmap="RdBu", vmin=vmin, vmax=vmax)